# 04. 모델 비교와 선정

**과제 조건: 1 에폭 최대 3분.**

순서가 중요하다. 모델을 먼저 고르고 시간을 재면, 조건을 못 맞췄을 때 처음부터 다시 해야 한다.
**시간 예산을 먼저 재고, 통과한 후보 안에서만 고른다.**

| 절 | 내용 |
|---|---|
| 4-1 | 속도 프로브 — 학습 전에 1 에폭 시간을 예측한다 |
| 4-2 | 통과한 후보들을 같은 조건에서 학습해 비교 |
| 4-3 | 성능 vs 비용으로 최종 선정 |

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch

import wbc
wbc.use_korean_font()

cfg = wbc.load_cfg()
PRESET     = cfg['preset']        # 03 에서 고른 전처리
IMAGE_SIZE = 224
BATCH_SIZE = 64
SCREEN_EPOCHS = 8
SUBSET     = None
wbc.NUM_WORKERS = 4

N_TRAIN = int(9957 * 0.8)
print('전처리:', PRESET, '| 학습 장수:', N_TRAIN, '| 장치:', wbc.device)

## 4-1. 속도 프로브

실제 학습 없이 **더미 배치 몇 개만 흘려** 초당 처리량(img/s)을 재고,
학습 장수를 곱해 1 에폭 예상 시간을 계산한다 (검증·데이터로딩 몫으로 25% 가산).

> **왜 이게 필요한가**: 수업 5장에서 ResNetV2-50 은 4,173장에 200초/에폭이었다.
> 우리 학습셋은 약 7,966장으로 **거의 두 배**다. 같은 속도라면 380초 → **조건 위반**.
> 그래서 "수업에서 쓴 모델"을 그대로 가져오면 안 되고, 재보고 결정해야 한다.

In [ ]:
cands = [
    ('simplecnn',              224, 64),
    ('resnet18',               224, 64),
    ('resnet18',               160, 64),
    ('efficientnet_b0',        224, 64),
    ('mobilenetv3_small_100',  224, 64),
    ('resnet34',               224, 64),
    ('resnetv2_50',            224, 64),
]
rows = []
for name, size, bs in cands:
    try:
        rows.append(wbc.speed_probe(name, image_size=size, batch_size=bs, n_batches=10, n_train=N_TRAIN))
    except RuntimeError as e:
        print(f'{name} bs={bs} 실패 -> bs 32 로 재시도 ({str(e)[:60]})')
        rows.append(wbc.speed_probe(name, image_size=size, batch_size=32, n_batches=10, n_train=N_TRAIN))
probe = pd.DataFrame(rows)
probe['판정'] = np.where(probe['fits'], '○ 3분 이내', '✗ 초과')
display(probe[['model', 'image_size', 'batch_size', 'params_M', 'img_per_sec', 'est_epoch_sec', '판정']])

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.6))
lab = probe['model'] + '\n' + probe['image_size'].astype(str) + 'px'
ax.bar(lab, probe['est_epoch_sec'], color=['tab:green' if f else 'tab:red' for f in probe['fits']])
ax.axhline(180, ls='--', c='k'); ax.text(0, 186, '3분 예산', fontsize=9)
ax.set_ylabel('1 에폭 예상 시간 (초)'); ax.set_title('모델별 에폭 시간 — 과제 조건 대비')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

### 3분을 못 맞출 때의 대응 순서

1. **AMP(혼합정밀)** — `train_model` 에서 GPU면 이미 켜져 있다. 가장 싸다
2. **이미지 224 → 160** — 연산량이 약 절반
3. **더 가벼운 백본**
4. **배치 크기 조정** / `NUM_WORKERS` 를 올려 데이터 로딩 병목 제거
5. (최후) 학습셋 일부만 사용 — 성능이 떨어지므로 마지막 수단

**초록색만 후보다.** 빨간색은 성능이 아무리 좋아도 이번 과제에서는 쓸 수 없다.

## 4-2. 후보 모델 비교

**조건을 하나만 바꾼다** — 전처리·학습률·에폭·시드를 전부 고정하고 **백본만** 바꾼다.

In [ ]:
MODELS = [m for m in ['resnet18', 'resnet34', 'efficientnet_b0', 'mobilenetv3_small_100', 'resnetv2_50']
          if bool(probe.loc[probe.model == m, 'fits'].max())]
est = sum(float(probe.loc[probe.model == m, 'est_epoch_sec'].iloc[0]) for m in MODELS) * SCREEN_EPOCHS
print('비교 대상:', MODELS)
print(f'예상 총 소요시간 ≈ {est/60:.0f}분 (조기종료로 더 짧아질 수 있음)')

In [ ]:
for m in MODELS:
    wbc.run_experiment(f'M_{m}', model_name=m, preset=PRESET, image_size=IMAGE_SIZE,
                       batch_size=BATCH_SIZE, lr=3e-4, epochs=SCREEN_EPOCHS,
                       pretrained=True, patience=3, seed=42, subset=SUBSET)
    print('-' * 78)

In [ ]:
t = wbc.runs_table()
cmp_ = t[t.run_id.str.startswith('M_')].sort_values('val_macro_f1', ascending=False)
display(cmp_[['run_id', 'params_M', 'epoch_sec', 'best_epoch',
              'val_accuracy', 'val_macro_f1', 'val_auc']].round(4))

## 4-3. 성능 vs 비용 — 최종 선정

선정 기준은 **성능 하나가 아니라 셋**이다.

1. **3분 예산 준수** (필수 조건)
2. **검증 macro-F1** (성능)
3. **비용 대비 이득** — 파라미터가 2배인데 F1 이 0.002 오른다면 이득이 아니다.
   05 에서 하이퍼파라미터를 수십 번 돌려야 하므로 **속도가 곧 실험 횟수**다.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].barh(cmp_.run_id.str[2:], cmp_.val_macro_f1, color='tab:blue')
ax[0].set_xlim(max(0, cmp_.val_macro_f1.min() - 0.05), 1.0)
ax[0].set_xlabel('검증 macro-F1'); ax[0].set_title('성능')
ax[1].scatter(cmp_.epoch_sec, cmp_.val_macro_f1, s=60)
for _, r in cmp_.iterrows():
    ax[1].annotate(r.run_id[2:], (r.epoch_sec, r.val_macro_f1), fontsize=8,
                   xytext=(4, 4), textcoords='offset points')
ax[1].axvline(180, ls='--', c='r'); ax[1].set_xlabel('에폭당 실측 시간(초)')
ax[1].set_ylabel('검증 macro-F1'); ax[1].set_title('성능 vs 비용 (빨간선 = 3분)'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
cand = cmp_[cmp_.epoch_sec <= wbc.EPOCH_BUDGET_SEC].copy()
top = cand.val_macro_f1.max()
near = cand[cand.val_macro_f1 >= top - 0.005]        # 최고와 0.5%p 이내면 동급으로 본다
best = near.sort_values('epoch_sec').iloc[0]

print('3분 예산 통과 :', cand.run_id.str[2:].tolist())
print(f'최고 성능     : {cmp_.iloc[0].run_id[2:]} (macro-F1 {top:.4f})')
print(f'최종 선정     : {best.run_id[2:]} (macro-F1 {best.val_macro_f1:.4f}, '
      f'에폭 {best.epoch_sec:.0f}s, {best.params_M}M)')
print()
print('선정 이유')
print('  1) 3분 예산을 지킨다')
print('  2) 최고 성능과 0.5%p 이내 — 이 차이가 진짜인지는 08 McNemar 검정으로 확인한다')
print('  3) 그 안에서 가장 빠르다 -> 05 의 반복 실험 횟수를 늘려준다')

wbc.save_cfg(model_name=best.run_id[2:], image_size=IMAGE_SIZE, batch_size=BATCH_SIZE)

## 04 정리

- 학습 **전에** 속도를 재서 3분 예산으로 후보를 걸렀다
- 통과한 후보를 같은 조건에서 비교하고, 성능-비용 관점에서 골랐다
- 선택을 `config.json` 에 저장했다

> 최고 성능 모델과 선정 모델의 차이가 통계적으로 유의하지 **않다**는 것을
> 08 에서 확인하면, "더 싼 모델로 같은 성능을 냈다"는 강한 결론이 된다.

→ 다음: **05_하이퍼파라미터_최적화.ipynb**